# Modul 5: SQL-Abfragen — direkt lauffähig / SQL Queries — Directly Runnable

**DE:** Dieses Notebook führt die drei SQL-Skripte aus `sql/` gegen die im Repo enthaltene, bereits befüllte Datenbank `sql/retailco.db` aus (SQLite, geladen aus `purchase_orders.csv`). Es ist **kein Datenbank-Setup nötig** — einfach "Run All" ausführen. Ergebnisse sind unten auch ohne eigene Ausführung sichtbar (z. B. direkt auf GitHub).

**EN:** This notebook runs the three SQL scripts from `sql/` against the pre-loaded database `sql/retailco.db` already included in the repo (SQLite, loaded from `purchase_orders.csv`). **No database setup is required** — just "Run All". Results below are visible even without running the notebook yourself (e.g. directly on GitHub).

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path('..') / 'sql' / 'retailco.db'
con = sqlite3.connect(DB_PATH)
print('Connected:', DB_PATH.resolve())
print('Rows in purchase_orders:', pd.read_sql('SELECT COUNT(*) AS n FROM purchase_orders', con)['n'][0])

Connected: /home/claude/retailco/Retailco-finance-procurement-controlling/sql/retailco.db
Rows in purchase_orders: 900


## 1. Data Quality / Datenqualität

**DE:** Kritische Prüfungen aus `01_data_quality.sql` — alle Checks müssen 0 Treffer liefern.

**EN:** Critical checks from `01_data_quality.sql` — every check should return 0 hits.

In [2]:
queries_01 = {
    'Duplicate PO IDs': """SELECT PO_ID, COUNT(*) AS row_count FROM purchase_orders GROUP BY PO_ID HAVING COUNT(*) > 1""",
    'Missing Supplier IDs': """SELECT COUNT(*) AS missing_supplier_ids FROM purchase_orders WHERE Supplier_ID IS NULL OR TRIM(Supplier_ID) = ''""",
    'Invalid Quantities': """SELECT COUNT(*) AS invalid_quantities FROM purchase_orders WHERE Quantity <= 0""",
    'Invalid Prices': """SELECT COUNT(*) AS invalid_prices FROM purchase_orders WHERE Actual_Unit_Price <= 0 OR Planned_Unit_Price <= 0""",
    'Invalid Dates': """SELECT COUNT(*) AS invalid_dates FROM purchase_orders WHERE PO_Date IS NULL""",
}

for label, q in queries_01.items():
    result = pd.read_sql(q, con)
    status = 'PASS' if (result.empty or (result.iloc[0] == 0).all()) else 'REVIEW'
    print(f'{label:<22} -> {status}')
    display(result)

Duplicate PO IDs       -> PASS


,PO_ID,row_count


Missing Supplier IDs   -> PASS


,missing_supplier_ids
0,0


Invalid Quantities     -> PASS


,invalid_quantities
0,0


Invalid Prices         -> PASS


,invalid_prices
0,0


Invalid Dates          -> PASS


,invalid_dates
0,0


## 2. Procurement KPIs / Einkaufs-KPIs

**DE:** Zentrale Kennzahlen und monatlicher Trend aus `02_procurement_kpis.sql`.

**EN:** Core KPIs and monthly trend from `02_procurement_kpis.sql`.

In [3]:
kpi_query = """
SELECT
    SUM(Quantity * Actual_Unit_Price) AS purchase_spend,
    SUM(Quantity * Planned_Unit_Price) AS planned_spend,
    SUM(Quantity * (Actual_Unit_Price - Planned_Unit_Price)) AS purchase_price_variance,
    COUNT(DISTINCT Supplier_ID) AS supplier_count,
    COUNT(DISTINCT PO_ID) AS purchase_order_count,
    AVG(Quantity * Actual_Unit_Price) AS average_po_value
FROM purchase_orders;
"""
display(pd.read_sql(kpi_query, con).round(2))

monthly_query = """
SELECT
    SUBSTR(PO_Date, 1, 7) AS year_month,
    SUM(Quantity * Actual_Unit_Price) AS purchase_spend,
    SUM(Quantity * Planned_Unit_Price) AS planned_spend,
    SUM(Quantity * (Actual_Unit_Price - Planned_Unit_Price)) AS ppv
FROM purchase_orders
GROUP BY SUBSTR(PO_Date, 1, 7)
ORDER BY year_month;
"""
display(pd.read_sql(monthly_query, con).round(2))

,purchase_spend,planned_spend,purchase_price_variance,supplier_count,purchase_order_count,average_po_value
0,4824720.92,4771530.23,53190.69,12,900,5360.8


,year_month,purchase_spend,planned_spend,ppv
0,2015-01,159490.55,163809.86,-4319.31
1,2015-02,120119.39,119891.34,228.05
2,2015-03,581242.37,574674.89,6567.48
3,2015-04,819352.61,793769.34,25583.27
4,2015-05,665217.05,653073.43,12143.62
5,2015-06,452444.44,466845.53,-14401.09
6,2015-07,270899.58,272947.17,-2047.59
7,2015-08,647150.40,637537.27,9613.13
8,2015-09,145788.19,145849.29,-61.10
9,2015-10,489089.51,477950.92,11138.59


## 3. Supplier &amp; Category Analysis / Lieferanten- &amp; Warengruppenanalyse

**DE:** Aggregation aus `03_supplier_analysis.sql`.

**EN:** Aggregation from `03_supplier_analysis.sql`.

In [4]:
supplier_query = """
SELECT
    Supplier_ID,
    Supplier_Name,
    SUM(Quantity * Actual_Unit_Price) AS purchase_spend,
    SUM(Quantity * Planned_Unit_Price) AS planned_spend,
    SUM(Quantity * (Actual_Unit_Price - Planned_Unit_Price)) AS ppv,
    COUNT(DISTINCT PO_ID) AS purchase_orders
FROM purchase_orders
GROUP BY Supplier_ID, Supplier_Name
ORDER BY purchase_spend DESC;
"""
display(pd.read_sql(supplier_query, con).round(2))

category_query = """
SELECT
    Category,
    SUM(Quantity * Actual_Unit_Price) AS purchase_spend,
    SUM(Quantity * Planned_Unit_Price) AS planned_spend,
    SUM(Quantity * (Actual_Unit_Price - Planned_Unit_Price)) AS ppv
FROM purchase_orders
GROUP BY Category
ORDER BY purchase_spend DESC;
"""
display(pd.read_sql(category_query, con).round(2))

con.close()

,Supplier_ID,Supplier_Name,purchase_spend,planned_spend,ppv,purchase_orders
0,SUP008,Supplier H,988042.42,961003.09,27039.33,61
1,SUP004,Supplier D,930909.21,889721.49,41187.72,91
2,SUP010,Supplier J,431501.33,424189.44,7311.89,44
3,SUP003,Supplier C,371570.53,386503.63,-14933.10,106
4,SUP002,Supplier B,365157.68,371345.44,-6187.76,108
5,SUP006,Supplier F,338849.74,345243.30,-6393.56,93
6,SUP001,Supplier A,325844.50,322810.62,3033.88,112
7,SUP005,Supplier E,288277.69,284894.78,3382.91,90
8,SUP012,Supplier L,263141.41,257902.73,5238.68,28
9,SUP011,Supplier K,206698.46,202498.43,4200.03,40


,Category,purchase_spend,planned_spend,ppv
0,Bikes,3338375.30,3291904.02,46471.28
1,Accessories,767261.08,764364.64,2896.44
2,Clothing,719084.54,715261.57,3822.97
